### Imports

In [1]:
from collections import deque
import heapq
from itertools import combinations, chain

### Modelagem

In [2]:
# Tempos das pessoas
TEMPOS = {
    'A': 1,
    'B': 2,
    'C': 5,
    'D': 10
}

# Todas as pessoas
PESSOAS = {'A', 'B', 'C', 'D'}

# Estado = (conjunto de quem está no lado inicial, lado da tocha)
estado_inicial = (set(PESSOAS), 'inicio')
estado_objetivo = (set(), 'final')

# Função para inverter o lado da tocha
def lado_oposto(lado):
    return 'final' if lado == 'inicio' else 'inicio'

# Mostrar o estado de forma legível
def mostrar_estado(estado):
    lado_inicio, tocha = estado
    lado_final = PESSOAS - lado_inicio
    print(f"Início: {sorted(lado_inicio)} | Final: {sorted(lado_final)} | Tocha: {tocha}")

# Estado inicial e final
mostrar_estado(estado_inicial)
mostrar_estado(estado_objetivo)


Início: ['A', 'B', 'C', 'D'] | Final: [] | Tocha: inicio
Início: [] | Final: ['A', 'B', 'C', 'D'] | Tocha: final


### Função Sucessora

In [3]:
def construir_sucessores(estado):
    # Gera todos sucessores possiveis de um estado
    # retorna tuplas: (estado_novo, pessoas_movidas, custo)
    lado_inicio, tocha = estado
    sucessores = []

    if tocha == 'inicio':
        for grupo in chain(combinations(lado_inicio, 1), combinations(lado_inicio, 2)):
            grupo = set(grupo)
            novo_lado_inicio = lado_inicio - grupo
            novo_estado = (novo_lado_inicio, 'final')
            custo = max(TEMPOS[p] for p in grupo)
            sucessores.append((novo_estado, grupo, custo))

    else:  # tocha está no lado 'final'
        lado_final = PESSOAS - lado_inicio
        for pessoa in lado_final:
            grupo = {pessoa}
            novo_lado_inicio = lado_inicio | grupo
            novo_estado = (novo_lado_inicio, 'inicio')
            custo = TEMPOS[pessoa]
            sucessores.append((novo_estado, grupo, custo))

    return sucessores

def mostrar_caminho(caminho, custo_total):
    for i, (estado, grupo, custo) in enumerate(caminho):
        print(f"Passo {i}:")
        mostrar_estado(estado)
        if grupo:
            print(f"Ação: {sorted(grupo)} | Custo da ação: {custo}")
        print("---")
    print(f"Custo total da travessia: {custo_total} minutos")

In [4]:
print("Sucessores do estado inicial:")
sucessores = construir_sucessores(estado_inicial)
for novo_estado, grupo, custo in sucessores:
    print(f"Ação: {sorted(grupo)} | Custo: {custo}")
    mostrar_estado(novo_estado)
    print("---")


Sucessores do estado inicial:
Ação: ['C'] | Custo: 5
Início: ['A', 'B', 'D'] | Final: ['C'] | Tocha: final
---
Ação: ['B'] | Custo: 2
Início: ['A', 'C', 'D'] | Final: ['B'] | Tocha: final
---
Ação: ['D'] | Custo: 10
Início: ['A', 'B', 'C'] | Final: ['D'] | Tocha: final
---
Ação: ['A'] | Custo: 1
Início: ['B', 'C', 'D'] | Final: ['A'] | Tocha: final
---
Ação: ['B', 'C'] | Custo: 5
Início: ['A', 'D'] | Final: ['B', 'C'] | Tocha: final
---
Ação: ['C', 'D'] | Custo: 10
Início: ['A', 'B'] | Final: ['C', 'D'] | Tocha: final
---
Ação: ['A', 'C'] | Custo: 5
Início: ['B', 'D'] | Final: ['A', 'C'] | Tocha: final
---
Ação: ['B', 'D'] | Custo: 10
Início: ['A', 'C'] | Final: ['B', 'D'] | Tocha: final
---
Ação: ['A', 'B'] | Custo: 2
Início: ['C', 'D'] | Final: ['A', 'B'] | Tocha: final
---
Ação: ['A', 'D'] | Custo: 10
Início: ['B', 'C'] | Final: ['A', 'D'] | Tocha: final
---


### Métodos de busca não informada

In [5]:
def bfs(estado_inicial, estado_objetivo):
    fila = deque()
    fila.append((estado_inicial, [], 0))
    visitados = set()
    # marque o inicial imediatamente
    visitados.add((frozenset(estado_inicial[0]), estado_inicial[1]))

    while fila:
        estado_atual, caminho, custo_atual = fila.popleft()

        if estado_atual == estado_objetivo:
            return caminho + [(estado_atual, None, 0)], custo_atual

        for novo_estado, grupo, custo in construir_sucessores(estado_atual):
            novo_hash = (frozenset(novo_estado[0]), novo_estado[1])
            if novo_hash not in visitados:
                visitados.add(novo_hash)  # marca ao enfileirar
                novo_caminho = caminho + [(estado_atual, grupo, custo)]
                fila.append((novo_estado, novo_caminho, custo_atual + custo))

    return None, None

def dfs(estado_inicial, estado_objetivo, limite=10_000):
    pilha = [(estado_inicial, [], 0)]
    visitados = set()
    # marque o inicial imediatamente
    visitados.add((frozenset(estado_inicial[0]), estado_inicial[1]))
    passos = 0

    while pilha and passos < limite:
        estado_atual, caminho, custo_atual = pilha.pop()
        passos += 1

        if estado_atual == estado_objetivo:
            return caminho + [(estado_atual, None, 0)], custo_atual

        for novo_estado, grupo, custo in construir_sucessores(estado_atual):
            novo_hash = (frozenset(novo_estado[0]), novo_estado[1])
            if novo_hash not in visitados:
                visitados.add(novo_hash)  # marca ao empilhar
                novo_caminho = caminho + [(estado_atual, grupo, custo)]
                pilha.append((novo_estado, novo_caminho, custo_atual + custo))

    return None, None


In [6]:
caminho_bfs, custo_bfs = bfs(estado_inicial, estado_objetivo)
print("BFS (Busca em Largura):")
mostrar_caminho(caminho_bfs, custo_bfs)

print("\nDFS (Busca em Profundidade):")
caminho_dfs, custo_dfs = dfs(estado_inicial, estado_objetivo)
mostrar_caminho(caminho_dfs, custo_dfs)


BFS (Busca em Largura):
Passo 0:
Início: ['A', 'B', 'C', 'D'] | Final: [] | Tocha: inicio
Ação: ['B', 'C'] | Custo da ação: 5
---
Passo 1:
Início: ['A', 'D'] | Final: ['B', 'C'] | Tocha: final
Ação: ['C'] | Custo da ação: 5
---
Passo 2:
Início: ['A', 'C', 'D'] | Final: ['B'] | Tocha: inicio
Ação: ['C', 'D'] | Custo da ação: 10
---
Passo 3:
Início: ['A'] | Final: ['B', 'C', 'D'] | Tocha: final
Ação: ['C'] | Custo da ação: 5
---
Passo 4:
Início: ['A', 'C'] | Final: ['B', 'D'] | Tocha: inicio
Ação: ['A', 'C'] | Custo da ação: 5
---
Passo 5:
Início: [] | Final: ['A', 'B', 'C', 'D'] | Tocha: final
---
Custo total da travessia: 30 minutos

DFS (Busca em Profundidade):
Passo 0:
Início: ['A', 'B', 'C', 'D'] | Final: [] | Tocha: inicio
Ação: ['A', 'D'] | Custo da ação: 10
---
Passo 1:
Início: ['B', 'C'] | Final: ['A', 'D'] | Tocha: final
Ação: ['A'] | Custo da ação: 1
---
Passo 2:
Início: ['A', 'B', 'C'] | Final: ['D'] | Tocha: inicio
Ação: ['A', 'B'] | Custo da ação: 2
---
Passo 3:
Início: ['C

### Métodos de busca informada

In [ ]:
# heuristica baseada no tempo do mais lento que ainda está no lado inicial
# logo o custo restante nunca pode ser menor que esse tempo
def heuristica_max_lento(estado):
    inicio, _ = estado
    if not inicio:
        return 0
    return max(TEMPOS[p] for p in inicio)

def custo_uniforme(estado_inicial, estado_objetivo):
    # heap: (g, estado, caminho)
    heap = []
    heapq.heappush(heap, (0, estado_inicial, []))
    melhor_g = { (frozenset(estado_inicial[0]), estado_inicial[1]) : 0 }

    while heap:
        g_atual, estado, caminho = heapq.heappop(heap)
        if estado == estado_objetivo:
            return caminho + [(estado, None, 0)], g_atual

        for novo_estado, grupo, custo in construir_sucessores(estado):
            g_novo = g_atual + custo
            chave = (frozenset(novo_estado[0]), novo_estado[1])
            if chave not in melhor_g or g_novo < melhor_g[chave]:
                melhor_g[chave] = g_novo
                heapq.heappush(heap, (g_novo, novo_estado, caminho + [(estado, grupo, custo)]))
    return None, None

def a_estrela(estado_inicial, estado_objetivo, h_fn=heuristica_max_lento):
    # heap: (f=g+h, g, estado, caminho)
    heap = []
    g0 = 0
    f0 = g0 + h_fn(estado_inicial)
    heapq.heappush(heap, (f0, g0, estado_inicial, []))
    melhor_g = { (frozenset(estado_inicial[0]), estado_inicial[1]) : 0 }

    while heap:
        f_atual, g_atual, estado, caminho = heapq.heappop(heap)

        if f_atual != g_atual + h_fn(estado):
            continue

        if estado == estado_objetivo:
            return caminho + [(estado, None, 0)], g_atual

        for novo_estado, grupo, custo in construir_sucessores(estado):
            g_novo = g_atual + custo
            chave = (frozenset(novo_estado[0]), novo_estado[1])
            if chave not in melhor_g or g_novo < melhor_g[chave]:
                melhor_g[chave] = g_novo
                f_novo = g_novo + h_fn(novo_estado)
                heapq.heappush(heap, (f_novo, g_novo, novo_estado, caminho + [(estado, grupo, custo)]))
    return None, None

In [8]:
print("Custo Uniforme:")
cam_cu, custo_cu = custo_uniforme(estado_inicial, estado_objetivo)
mostrar_caminho(cam_cu, custo_cu)

print("\nA*:")
cam_a, custo_a = a_estrela(estado_inicial, estado_objetivo)
mostrar_caminho(cam_a, custo_a)


Custo Uniforme:
Passo 0:
Início: ['A', 'B', 'C', 'D'] | Final: [] | Tocha: inicio
Ação: ['A', 'B'] | Custo da ação: 2
---
Passo 1:
Início: ['C', 'D'] | Final: ['A', 'B'] | Tocha: final
Ação: ['A'] | Custo da ação: 1
---
Passo 2:
Início: ['A', 'C', 'D'] | Final: ['B'] | Tocha: inicio
Ação: ['C', 'D'] | Custo da ação: 10
---
Passo 3:
Início: ['A'] | Final: ['B', 'C', 'D'] | Tocha: final
Ação: ['B'] | Custo da ação: 2
---
Passo 4:
Início: ['A', 'B'] | Final: ['C', 'D'] | Tocha: inicio
Ação: ['A', 'B'] | Custo da ação: 2
---
Passo 5:
Início: [] | Final: ['A', 'B', 'C', 'D'] | Tocha: final
---
Custo total da travessia: 17 minutos

A*:
Passo 0:
Início: ['A', 'B', 'C', 'D'] | Final: [] | Tocha: inicio
Ação: ['A', 'B'] | Custo da ação: 2
---
Passo 1:
Início: ['C', 'D'] | Final: ['A', 'B'] | Tocha: final
Ação: ['A'] | Custo da ação: 1
---
Passo 2:
Início: ['A', 'C', 'D'] | Final: ['B'] | Tocha: inicio
Ação: ['C', 'D'] | Custo da ação: 10
---
Passo 3:
Início: ['A'] | Final: ['B', 'C', 'D'] | Toc